# Week 10 Homework: ARIA v7.0 - The All-Weather Auditor

**Course:** Remote Sensing & Spatial Information Analysis  
**Student:** Wade  
**Case:** Hualien, Typhoon Fung-wong  
**Objective:** integrate Sentinel-1 SAR, Sentinel-2 NDWI/cloud masking, and DEM slope auditing to detect flood candidates under cloud cover.

**Captain's Log:** The assignment expected a local `S1_Hualien_dB.tif`, but it was not available in the provided folder. I therefore used the Week 10 STAC workflow to stream Sentinel-1 RTC from Microsoft Planetary Computer. This is documented because data provenance is part of the audit trail.


## Executive Summary

| Metric                                | Value      |
|:--------------------------------------|:-----------|
| SAR acquisition                       | 2025-09-18 |
| SAR orbit                             | ascending  |
| SAR threshold                         | -18 dB     |
| NDWI threshold                        | 0.0        |
| Cloud cover in selected optical scene | 100.0%     |
| SAR flood-like water pixels           | 1,038      |
| SAR flood-like water area             | 0.934 km2  |
| Mean flood-zone VV backscatter        | -19.62 dB  |
| High-confidence dual-sensor area      | 0.000 km2  |
| SAR-only cloudy area                  | 0.934 km2  |
| Steep-slope false positives flagged   | 0.613 km2  |
| Post-audit flood candidate area       | 0.321 km2  |

**Main finding:** under a 100.0% cloud-covered Sentinel-2 scene, optical confirmation is unavailable. ARIA v7.0 therefore reports **0.934 km2** as SAR-only cloudy flood candidates, then flags **0.613 km2** of steep-slope detections for manual review. The post-audit candidate area is approximately **0.321 km2**.


## Reproducibility Note

The workflow is reproducible with:

```bash
python generate_week10_submission.py
```

The script regenerates the figures, CSV tables, `.env`, AI briefing markdown, and this notebook. I kept `.env` local and uploaded `.env.example` to GitHub to follow the homework instruction not to commit environment files.


## Task 1: SAR All-Weather Flood Detection

### Method

1. Load Sentinel-1 RTC VV backscatter and convert linear sigma-nought to dB.
2. Apply a 5 x 5 median filter before thresholding to suppress speckle.
3. Use `VV < -18 dB` as the conservative ARIA default water threshold.
4. Apply morphological opening and connected-component filtering to remove tiny fragments.
5. Convert pixels to area using the 30 m working grid.

### Required Result Table

| Item                               | Result                                                                                               |
|:-----------------------------------|:-----------------------------------------------------------------------------------------------------|
| Water pixels                       | 1,038                                                                                                |
| Flood-like water area              | 0.934 km2                                                                                            |
| Mean backscatter in SAR flood zone | -19.62 dB                                                                                            |
| Statement                          | SAR detected 0.934 km2 of flood-like water where optical evidence was blocked by 100.0% cloud cover. |

![Task 1 SAR Detection](week10_outputs/task1_sar_detection_panel.png)

### Discussion

The filtered SAR panel shows why thresholding raw SAR is risky: isolated dark speckle pixels can look like water. The median filter makes the flood mask spatially more coherent before thresholding. The mean flood-zone backscatter of **-19.62 dB** is consistent with smooth water surfaces producing low VV return. Because the selected optical scene is fully clouded, this SAR-derived layer is the only usable all-weather observation for this timestamp.


## Task 2: Sensor Fusion - Multi-Source Confidence Map

### Fusion Logic Used

| Optical NDWI water | SAR water | Cloud masked | Class |
|---|---|---|---|
| Yes | Yes | No | High Confidence |
| No or unusable | Yes | Yes | SAR Only (Cloudy) |
| Yes | No | No | Optical Only |
| No | No | Any | No Detection |

### Area Statistics

| class             |   code |   pixels |   area_km2 |
|:------------------|-------:|---------:|-----------:|
| No Detection      |      0 |   616810 |    555.129 |
| Optical Only      |      1 |        0 |      0     |
| SAR Only (Cloudy) |      2 |     1038 |      0.934 |
| High Confidence   |      3 |        0 |      0     |

![Task 2 Confidence Map](week10_outputs/task2_confidence_map.png)

### Interpretation

The high-confidence class is **0.000 km2** because optical evidence is unusable under **100.0% cloud cover**. This is expected, not a workflow failure. The important result is that SAR still identifies **0.934 km2** of flood-like water in the cloudy scene. In an emergency workflow, this class should be treated as a reconnaissance and field-verification priority rather than ignored because optical data are missing.


## Task 3: Topographic Audit - DEM and Slope Assessment

### False Positives Flagged by Slope Class

| slope_class   |   removed_pixels |   removed_km2 |
|:--------------|-----------------:|--------------:|
| 25-35 deg     |              104 |         0.094 |
| 35-45 deg     |              108 |         0.097 |
| >45 deg       |              469 |         0.422 |

![Task 3 Topographic Audit](week10_outputs/task3_topographic_audit.png)

### Discussion: Is DEM Slope Appropriate Here?

For Hualien floodplain-style inundation, a DEM slope filter is appropriate because standing water should not persist on slopes steeper than **25 degrees**. The audit flagged **0.613 km2** of SAR water detections on steep terrain, which are more likely radar shadow, layover, or dark slope artifacts than true floodwater.

This rule should be applied more cautiously in landslide or barrier-lake cases. If the disaster itself changed the topography, an older DEM may describe pre-event terrain and could incorrectly remove valid post-event water. In that situation I would combine slope screening with morphological opening, connected-component filtering, same-orbit SAR differencing, and updated UAV/LiDAR/InSAR elevation data.


## AI Strategic Briefing

### Exact Prompt

```text
You are an emergency management advisor for Hualien County during Typhoon Fung-wong.
Based on these ARIA v7.0 sensor fusion results, generate a strategic operational briefing that covers:
1. Which areas require immediate evacuation?
2. How should resources be allocated between high-confidence and SAR-only zones?
3. What are the limitations of the current assessment?
4. What additional data would improve confidence?

Metrics:
- High confidence flood area: 0.000 km2
- SAR-only cloudy flood area: 0.934 km2
- False positives removed by topographic filter: 0.613 km2
- Cloud cover percentage: 100.0%
- SAR threshold: -18 dB, chosen as the ARIA default for conservative SAR flood extraction
- NDWI threshold: 0.0, chosen for turbid storm water rather than clear-water NDWI ~0.3
```

### LLM Response

Because the selected optical scene is 100.0% cloud-covered, there are no dual-sensor high-confidence flood pixels in this run. The operational priority should therefore be rapid field verification of the 0.934 km2 SAR-only cloudy zone rather than automatic evacuation based on optical confirmation.

For resource allocation, dispatch reconnaissance teams, UAVs, road patrols, and gauge checks first to SAR-only clusters near settlements, roads, bridges, and low-lying drainage corridors. Heavy evacuation or rescue assets should be staged nearby but committed after field reports or follow-up SAR/optical passes confirm persistent water.

The main limitations are SAR speckle, threshold sensitivity, radar shadow or layover on steep terrain, and the absence of optical confirmation under complete cloud cover. The slope audit flagged 0.613 km2 as likely steep-terrain false positives, leaving about 0.321 km2 as post-audit flood candidates.

Confidence would improve with river gauge records, disaster reports, UAV imagery, road closure data, settlement and road overlays, and a second Sentinel-1 acquisition from the same orbit.

### Reflection

The LLM response is useful because it does not pretend that zero high-confidence area means zero flood risk. It correctly treats SAR-only detections as a reconnaissance priority under complete cloud cover. The weakness is that the summary metrics do not contain village names, road segments, or population exposure, so the briefing cannot specify exact evacuation sites. I would use it as an incident-command triage memo and then overlay the map with roads, settlements, shelters, and live field reports.

## ARIA v7.0 vs. v6.0 Comparison

| Metric                           | W9 Optical Only              | W10 Fused                     | Improvement               |
|:---------------------------------|:-----------------------------|:------------------------------|:--------------------------|
| Total detected flood/change area | 33.346 km2                   | 0.934 km2                     | -32.412 km2               |
| Cloud-covered area analyzed      | 0 km2                        | 0.934 km2 SAR-only class      | cloud gaps audited by SAR |
| False positives handled          | phantom water removed by SCL | 0.613 km2 flagged by slope    | adds terrain audit        |
| Confidence levels                | 3-zone                       | 4-class + false-positive flag | finer triage              |

### Interpretation

W9 optical-only analysis mapped a broader disaster-change signal, including vegetation loss and debris-related spectral change. W10 is intentionally narrower: it asks whether SAR can detect flood-like water when the optical scene is clouded out. Therefore the decrease from W9's 33.346 km2 to W10's 0.934 km2 is not a performance failure; it reflects a stricter water-focused target and a more conservative SAR threshold.

The main improvement is operational rather than simply numerical: ARIA v7.0 can still produce an auditable flood candidate layer under 100.0% cloud cover. The topographic audit further separates physically plausible low-slope water from steep-terrain SAR artifacts.


## Output Verification and Sanity Checks

| Check | Result |
|---|---|
| Median filter before thresholding | Passed |
| SAR threshold documented | `-18 dB` |
| Flood mask covers only a small part of the BBOX | Passed: 0.934 km2 out of the analysis grid |
| Cloud-cover scenario matches assignment | Passed: 100.0% cloud cover |
| Fusion logic handles clouded optical data | Passed: high-confidence is 0, SAR-only cloudy is 0.934 km2 |
| Topographic audit performed | Passed: 0.613 km2 flagged |
| Output files saved | Figures, CSV tables, briefing markdown, and notebook are under `week10_outputs/` |

### Final Interpretation

ARIA v7.0 adds value precisely when optical satellites fail. In this run, W10 does not claim dual-sensor confirmation because the optical scene is cloud-covered. Instead, it produces a conservative SAR-only candidate layer and then uses slope to identify likely terrain artifacts. That is a more honest emergency product than forcing a high-confidence label where optical evidence does not exist.


## Submission Checklist

- [x] SAR flood extraction with median filtering
- [x] 2 x 2 SAR visualization panel
- [x] Flood area, pixel count, and mean backscatter table
- [x] Four-class fusion map and area statistics
- [x] DEM/slope topographic audit
- [x] False-positive table by slope class
- [x] AI strategic briefing with exact prompt, response, and reflection
- [x] W9 vs W10 comparison table
- [x] `.env.example` for reproducibility
- [x] Discussion and output verification included
